In [1]:
%load_ext autoreload
%autoreload 2

In [17]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch_geometric as pyg

from baseline_evals.feature_selection import variance_filtering

from torch_geometric.utils import to_networkx


from bipartite_gnn.train_test import GNNTrainer
from bipartite_gnn.model import GAT_2L, BipartiteRGAT, BiRGAT
from bipartite_gnn.graph_building import cosine_similarity_matrix, dense_to_coo
from bipartite_gnn.preprocessing import gg_interactions, get_mirna_gene_interactions, pp_interactions, gg_interactions
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo
from baseline_evals.feature_selection import class_variational_selection

In [18]:
# Check currently allocated CUDA memory in bytes
allocated_memory = torch.cuda.memory_allocated()
print(f"Currently allocated CUDA memory: {allocated_memory} bytes")

# Check maximum allocated CUDA memory in bytes
max_allocated_memory = torch.cuda.max_memory_allocated()
print(f"Maximum allocated CUDA memory: {max_allocated_memory} bytes")


Currently allocated CUDA memory: 0 bytes
Maximum allocated CUDA memory: 0 bytes


In [19]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnv.tsv", separator="\t", null_values=null_vals)
cna_th = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [20]:
cna.shape, cna_th.shape

((16711, 484), (24776, 484))

In [21]:
cna_th = cna_th.filter(
    pl.col("Gene Symbol").is_in(cna["Gene Symbol"])
)

In [22]:
assert mrna.columns[1:] == cna.columns[1:] == cna_th.columns[1:] == mirna.columns[1:], "Columns do not match"

In [23]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [24]:
mirna_gene_names = pl.read_csv("miRBaseConverter_Result.csv")
mirna_targets = mirna_gene_names["TargetName"].to_list()

In [40]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

random_state = 3

train_mask, val_test_mask = train_test_split(torch.arange(len(y)), test_size=0.4, stratify=y, random_state=random_state)
val_mask, test_mask = train_test_split(val_test_mask, test_size=0.5, random_state=random_state, stratify=y[val_test_mask])

# select top genes for mrna
select_mask_mrna, _ = class_variational_selection(mrna_X[train_mask], y[train_mask], 500)
select_mask_cna, _ = class_variational_selection(cna_X[train_mask], y[train_mask], 500)
select_mask_mirna, _ = class_variational_selection(mirna_X[train_mask], y[train_mask], 200)

mrna_X = mrna_X[:, select_mask_mrna]
cna_X = cna_X[:, select_mask_cna]

mrna_genes = np.array(mrna_gene_names)[select_mask_mrna]
cna_genes = np.array(cna_gene_names)[select_mask_cna]
 
mrna_A = create_diff_exp_connections_norm(mrna_X, train_mask, 2.4)
mirna_A = create_diff_exp_connections_norm(mirna_X, train_mask, 1.8)
cna_A = cna_th[:,1:].to_numpy().T[:, select_mask_cna].astype(np.float32)
cna_A = torch.tensor(cna_A, dtype=torch.float32)

print((cna_A.sum(dim=0) == 0).sum(), (cna_A.sum(dim=1) == 0).sum())
mrna_A_n = create_diff_exp_connections_nbnom(mrna_X, train_mask, 2.2)
mrna_A = torch.logical_or(mrna_A, mrna_A_n).int()

isolated sample nodes, isolated gene nodes, mean degree: 
tensor(9) tensor(2) tensor(12.2215, dtype=torch.float64)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(8) tensor(0) tensor(17.9607, dtype=torch.float64)
tensor(2) tensor(4)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(28) tensor(426) tensor(3.0663, dtype=torch.float64)


In [41]:
connected_nodes_mask = mrna_A.sum(dim=0) != 0
connected_nodes_mask_cna = cna_A.sum(dim=0) != 0

mrna_A = mrna_A[:, connected_nodes_mask]
mrna_X = mrna_X[:, connected_nodes_mask]

cna_A = cna_A[:, connected_nodes_mask_cna]
cna_X = cna_X[:, connected_nodes_mask_cna]

mrna_genes = mrna_genes[connected_nodes_mask]
cna_genes = cna_genes[connected_nodes_mask_cna]

print(mrna_A.shape, mrna_X.shape)

# normalize
std_scaler = StandardScaler()
std_scaler.fit(mrna_X.numpy()[train_mask])
mrna_X = torch.tensor(std_scaler.transform(mrna_X.numpy()))

std_scaler = StandardScaler()
std_scaler.fit(cna_X.numpy()[train_mask])
cna_X = torch.tensor(std_scaler.transform(cna_X.numpy()))

std_scaler = StandardScaler()
std_scaler.fit(mirna_X.numpy()[train_mask])
mirna_X = torch.tensor(std_scaler.transform(mirna_X.numpy()))


torch.Size([483, 498]) torch.Size([483, 498])


In [42]:
gg_A = gg_interactions(mrna_genes, mrna_genes)
pp_A = pp_interactions(mrna_genes, mrna_genes)

mrna_features_A = torch.logical_or(gg_A, pp_A).int()

gg_A = gg_interactions(cna_genes, cna_genes)
pp_A = pp_interactions(cna_genes, cna_genes)

cna_features_A = torch.logical_or(gg_A, pp_A).int()

mrna_features_A.sum(), cna_features_A.sum()

(tensor(750), tensor(482))

In [43]:
mirna_mrna_interactions = get_mirna_gene_interactions(mirna_targets, mrna_genes)
mirna_cna_interactions = get_mirna_gene_interactions(mirna_targets, cna_genes)

mc_A_gg = gg_interactions(mrna_genes, cna_genes)
mc_A_pp = pp_interactions(mrna_genes, cna_genes)

mc_A = torch.logical_or(mc_A_gg, mc_A_pp).int()

torch.Size([231, 498])
torch.Size([231, 498])


In [50]:
import torch_geometric.transforms as T

from bipartite_gnn.graph_building import dense_to_attributes

data = pyg.data.HeteroData()

proj_dim = 42

# sample node features
data["mrna"].x = mrna_X.float()
data["cna"].x = cna_X.float()
data["mirna"].x = mirna_X.float()

data.y = torch.tensor(y)

data.omics = ["mrna", "cna", "mirna"]
data.feature_names = ["mrna_feature", "cna_feature", "mirna_feature"]

# feature node projection
data["mrna_feature"].x = torch.zeros(mrna_X.shape[1], proj_dim)
data["cna_feature"].x = torch.zeros(cna_X.shape[1], proj_dim)
data["mirna_feature"].x = torch.zeros(mirna_X.shape[1], proj_dim)

# create edges
data["mrna", "diff_exp", "mrna_feature"].edge_index = dense_to_coo(mrna_A)
data["mrna", "diff_exp", "mrna_feature"].edge_attributes = dense_to_attributes(mrna_A)
data["mrna_feature", "interaction", "mrna_feature"].edge_index = dense_to_coo(mrna_features_A)

data["cna", "diff_exp", "cna_feature"].edge_index = dense_to_coo(cna_A)
data["cna", "diff_exp", "cna_feature"].edge_attributes = dense_to_attributes(cna_A)
data["cna_feature", "interaction", "cna_feature"].edge_index = dense_to_coo(cna_features_A)

# MIRNA INTERACTIONS
data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(mirna_A)
data["mirna", "diff_exp", "mirna_feature"].edge_attributes = dense_to_attributes(mirna_A)

data["mirna_feature", "interacts", "mrna_feature"].edge_index = dense_to_coo(mirna_mrna_interactions)
data["mirna_feature", "interacts", "cna_feature"].edge_attributes = dense_to_coo(mirna_cna_interactions)

data["mrna_feature", "interacts", "cna_feature"].edge_index = dense_to_coo(mc_A)

data = T.ToUndirected()(data)
# data = T.AddSelfLoops()(data)

# 2 for forward and backward diff exp, one for each self loop
data.num_relations = len(data.edge_index_dict.keys())
print("num_relations", data.num_relations)

data["train_mask"] = train_mask
data["val_mask"] = val_mask
data["test_mask"] = test_mask

data

num_relations 12


HeteroData(
  y=[483],
  omics=[3],
  feature_names=[3],
  num_relations=12,
  train_mask=[289],
  val_mask=[97],
  test_mask=[97],
  mrna={ x=[483, 498] },
  cna={ x=[483, 498] },
  mirna={ x=[483, 231] },
  mrna_feature={ x=[498, 42] },
  cna_feature={ x=[498, 42] },
  mirna_feature={ x=[231, 42] },
  (mrna, diff_exp, mrna_feature)={
    edge_index=[2, 6570],
    edge_attributes=[6570, 1],
  },
  (mrna_feature, interaction, mrna_feature)={ edge_index=[2, 1078] },
  (cna, diff_exp, cna_feature)={
    edge_index=[2, 125985],
    edge_attributes=[125985, 1],
  },
  (cna_feature, interaction, cna_feature)={ edge_index=[2, 712] },
  (mirna, diff_exp, mirna_feature)={
    edge_index=[2, 8675],
    edge_attributes=[8675, 1],
  },
  (mirna_feature, interacts, mrna_feature)={ edge_index=[2, 1280] },
  (mirna_feature, interacts, cna_feature)={ edge_attributes=[2, 887] },
  (mrna_feature, interacts, cna_feature)={ edge_index=[2, 754] },
  (mrna_feature, rev_diff_exp, mrna)={
    edge_index=[2, 

In [ ]:
from baseline_evals.mlp_eval import mlp_eval

mlp_eval(mrna_X_400, y, n_features=3000)

In [52]:
# model = BipartiteRGAT(
#     input_sizes=[mrna_X_400.shape[1]], # , cna_X_400.shape[1], mirna_X_200.shape[1]],
#     proj_dim=proj_dim,
#     hidden_channels=[128, 64], # num_layers = len(hidden_channels) - 1
#     num_labels=len(np.unique(y)),
#     num_relations=data.num_relations,
#     num_heads=1,
#     feature_integration_mode="linear",
# )

model = BiRGAT(
    omic_channels=data.omics,
    feature_names=data.feature_names,
    relations=list(data.edge_index_dict.keys()),
    input_dims=[mrna_X.shape[1], cna_X.shape[1], mirna_X.shape[1]],
    num_classes=len(np.unique(y)),
    proj_dim=proj_dim,
    hidden_channels=[86, 68, 46, 32],
    heads=4,
    dropout=0.4,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

ccounts = torch.unique(data.y[train_mask], return_counts=True)[1]
inv_w = 1.0 / ccounts.float()
class_weights = inv_w / inv_w.sum()

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
    optimizer=optimizer,
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15, min_lr=1e-5),
    params={
        "l1_lambda" : 0.001,
    }
)

trainer.train(
    data=data,
    epochs=150,
    log_interval=1,
)

Epoch: 001, 
Train Loss: 2.6469, Train Acc: 0.1799, Train F1 Macro: 0.1361, Train F1 Weighted: 0.1123
Val Loss: 1.3855, Val Acc: 0.1753, Val F1 Macro: 0.0746, Val F1 Weighted: 0.0523
Test Loss: 1.3849, Test Acc: 0.1856, Test F1 Macro: 0.0796, Test F1 Weighted: 0.0591
##################################################
Epoch: 002, 
Train Loss: 2.6390, Train Acc: 0.2941, Train F1 Macro: 0.2349, Train F1 Weighted: 0.2781
Val Loss: 1.3811, Val Acc: 0.2268, Val F1 Macro: 0.1455, Val F1 Weighted: 0.1478
Test Loss: 1.3805, Test Acc: 0.2577, Test F1 Macro: 0.1710, Test F1 Weighted: 0.1891
##################################################
Epoch: 003, 
Train Loss: 2.6390, Train Acc: 0.2976, Train F1 Macro: 0.2699, Train F1 Weighted: 0.3046
Val Loss: 1.3788, Val Acc: 0.1443, Val F1 Macro: 0.1338, Val F1 Weighted: 0.0872
Test Loss: 1.3776, Test Acc: 0.2268, Test F1 Macro: 0.2196, Test F1 Weighted: 0.1450
##################################################
Epoch: 004, 
Train Loss: 2.6359, Train Acc: